## Setup and Fetch Search Space

In [21]:
import json
import numpy as np
from urllib.request import Request, urlopen

TEAM_ID = "TEAM_58"
API_KEY = "oc_mivcH-R2ErVz8gVQuIRlVnEwQfULfaAk"
API_URL = "https://bftrxasgtunepckchcoz.supabase.co/functions/v1/oracle"


def api(payload):
    req = Request(
        API_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Content-Type": "application/json",
            "X-API-Key": API_KEY
        }
    )

    with urlopen(req, timeout=30) as response:
        return json.load(response)


# Get the assigned hyperparameter search space
spec = api({
    "action": "spec",
    "team_id": TEAM_ID
})

SPACE = spec["hyperparameters"]

print("Model:", spec["model"])

print("\nAssigned Hyperparameter Space:")

for name, values in SPACE.items():
    print(name, ":", values)

Model: RandomForestRegressor

Assigned Hyperparameter Space:
bootstrap : [True, False]
max_depth : [3, 4, 5, 7, 10, None]
max_features : [0.4, 0.6, 0.8, 1]
n_estimators : [3, 5, 8, 12]
min_samples_leaf : [1, 2, 4, 8]
min_samples_split : [2, 5, 10, 20]


### Oracle Query Function

In [22]:
def oracle_query(params):

    result = api({
        "action": "query",
        "team_id": TEAM_ID,
        "params": params
    })

    return float(result["loss"])

Oracle query function is ready.


### Store Results and Avoid Duplicate Calls

In [33]:
import pandas as pd

results = []
tested = set()


def make_key(params):

    return tuple(params[name] for name in SPACE.keys())


def test_config(params):

    key = make_key(params)

    if key in tested:
        return None

    loss = oracle_query(params)

    tested.add(key)

    result = params.copy()
    result["loss"] = loss

    results.append(result)

    return loss

### Starting Configuration

In [34]:
current_params = {
    "bootstrap": True,
    "max_depth": 5,
    "max_features": 0.8,
    "n_estimators": 8,
    "min_samples_leaf": 2,
    "min_samples_split": 5
}

current_loss = test_config(current_params)

print("Starting configuration:")
print(current_params)

print("\nStarting loss:")
print(current_loss)

Starting configuration:
{'bootstrap': True, 'max_depth': 5, 'max_features': 0.8, 'n_estimators': 8, 'min_samples_leaf': 2, 'min_samples_split': 5}

Starting loss:
8.08365219595768


### Find Neighbouring Values

In [35]:
def get_neighbours(params, parameter):

    values = SPACE[parameter]
    current_value = params[parameter]

    current_index = values.index(current_value)

    neighbours = []

    if current_index > 0:
        neighbours.append(values[current_index - 1])

    if current_index < len(values) - 1:
        neighbours.append(values[current_index + 1])

    return neighbours

### Adaptive Search

In [36]:
parameter_order = [
    "max_depth",
    "max_features",
    "n_estimators",
    "min_samples_leaf",
    "min_samples_split",
    "bootstrap"
]

best_params = current_params.copy()
best_loss = current_loss

improved = True

while improved:

    improved = False

    for parameter in parameter_order:

        neighbours = get_neighbours(best_params, parameter)

        for value in neighbours:

            new_params = best_params.copy()
            new_params[parameter] = value

            loss = test_config(new_params)

            if loss is None:
                continue

            if loss < best_loss:

                best_loss = loss
                best_params = new_params.copy()

                improved = True

                print(
                    "New best:",
                    round(best_loss, 6),
                    "|",
                    parameter,
                    "=",
                    value
                )

print("\nSearch completed.")

New best: 7.304889 | max_depth = 7
New best: 7.256361 | n_estimators = 12
New best: 7.089373 | min_samples_leaf = 1
New best: 7.034066 | n_estimators = 8

Search completed.


### View Tested Configurations

In [41]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values("loss").reset_index(drop=True)

print("Total Oracle Calls:", len(results_df))

display(results_df)

Total Oracle Calls: 25


,bootstrap,max_depth,max_features,n_estimators,min_samples_leaf,min_samples_split,loss
0,True,7,0.8,8,1,5,7.034066
1,True,7,0.8,12,1,5,7.089373
2,True,7,0.8,12,2,5,7.256361
3,True,7,0.8,8,2,5,7.304889
4,True,10,0.8,12,1,5,7.351841
5,True,10,0.8,8,1,5,7.388215
6,True,7,0.6,8,2,5,7.419415
7,True,7,0.6,8,1,5,7.529668
8,True,7,0.6,12,1,5,7.662882
9,True,7,0.8,12,4,5,7.677435


### Final Result

In [42]:

print("FINAL RESULT")

print("\nBest Hyperparameters:")

for parameter, value in best_params.items():
    print(parameter, ":", value)

print("\nOld Loss:")
print(current_loss)


print("\nBest Loss:")
print(best_loss)

print("\nTotal Oracle Calls:")
print(len(results))

FINAL RESULT

Best Hyperparameters:
bootstrap : True
max_depth : 7
max_features : 0.8
n_estimators : 8
min_samples_leaf : 1
min_samples_split : 5

Old Loss:
8.08365219595768

Best Loss:
7.03406600617746

Total Oracle Calls:
25
